# Lab 8 — Modeling at Summit Gear: specify, fit, interpret

You are an analyst at **Summit Gear Co.**, a B2B outdoor-equipment wholesaler. The
collections team has handed you one table of customers and asked you to answer two
questions:

1. **How long will an account take to pay?** — `days_to_pay`, a number. This is an **OLS
   regression**.
2. **Will an account default?** — `defaulted`, a yes / no. This is a **logistic
   regression**.

**The AI writes the Python; your job is to specify, validate, and interpret.** You pick the
target and the predictors, you confirm the model fit cleanly on the right data, and you
translate every number back into collections terms. The AI can fit a named model in
seconds — what it cannot do is decide *what* to fit or what the fit *means* for the
business. That judgment is the work, and it is the graded part. This notebook walks you
through it, one step at a time.

The shift from Week 7: there you wrote the code yourself; here you direct the AI to write
the model-fitting code and you own the specification, the validation, and the reading.

> **Start here:** run **Runtime > Run all** (Colab) or **Run All Cells** (Jupyter) now.
> Cells fail at first — the **Final check at the very bottom** lists exactly what is left
> and names the exercise to fix. Complete a piece, run all again, watch it turn green.

> **Use the AI tutor on the Week 8 lab page for every fitting step.** Fitting a named
> model is exactly what AI is good at. Writing the interpretation is the graded work, and
> the tutor will not write that for you until you have taken a first pass.

## How to run this notebook — two ways

Work this lab in whichever AI setup you already have:

- **Google Colab with Gemini.** Open the notebook in Colab and use the built-in Gemini
  assistant for the fitting steps. Nothing to install — `pandas`, `numpy`, and
  `statsmodels` are already there.
- **Locally with Claude.** If you have Claude, add the Claude extension to your browser or
  editor and work the downloaded notebook on your own machine (Anaconda or Jupyter).
  Install the packages once first — see Section 0.

Both the downloadable `.ipynb` and a one-click Colab link are provided *(Colab link on the
lab page)*. Pick the one that fits how you already work; the exercises are identical either
way.

## How to work this lab — the AI-first contract

Every fitting step follows the same three moves:

- **You specify.** In plain words, name the target and the predictor set you want, and
  why each predictor belongs and is known at decision time (avoid look-ahead bias).
- **The AI writes the code.** Ask the tutor for the `pandas` / `statsmodels` lines that
  fit exactly that model and print the output. Paste them into the fit cell.
- **You validate and interpret.** Run it. Confirm it fit on the right rows with the right
  predictors. Then read every number back in collections terms, in your own words.

You also write three small **helper functions** by hand — the arithmetic underneath the
fits — so you know what the AI's code is doing. Each is stubbed as `return None`; replace
the stub with the real body and a one-sentence docstring. A complete helper keeps its
`def` line exactly as given, has a docstring, and returns the right value:

```python
def amount_with_tax(amount, tax_rate):
    """Return the amount grossed up by the tax rate (0.0725 means 7.25 percent)."""
    return amount * (1 + tax_rate)
```

Keep every `def` line and every model-variable name exactly as given — the Final check
and the grader call them by name.

## How this notebook works

This notebook is the **modeling** work, and only that: three exercises — an **OLS**
regression for `days_to_pay`, a **correlation-matrix** collinearity check, and a
**logistic** regression for `defaulted` — each paired with a small **helper function** you
write by hand, and a single **testing function** at the very end. The cells you complete are
of three kinds: **fit cells** (paste the AI's model code and name the result exactly as
told), **helper cells** (replace `return None` with the body), and **markdown answer cells**
(write your interpretation). Run the **single Final check at the bottom** any time to see
what is left.

The HTML write-up and your written assessment are separate items in the Lab 8 quiz in
Canvas, not in this notebook.

## Section 0 — Setup

**In Google Colab** these packages are already installed — just run the cell below.
**Running locally** (Anaconda or Jupyter on your own computer) means installing them once
first:

```
pip install pandas numpy statsmodels scikit-learn
```

The packages you use: **`pandas`** (the data table), **`numpy`** (arithmetic), and
**`statsmodels`** (the regression fits). **`scikit-learn`** is optional — you only need it
if you choose its `LogisticRegression` in Exercise 3 instead of `statsmodels`.

In [6]:
# Section 0 — run once. In Colab these are preinstalled.
import pandas as pd
import numpy as np
import math
import statsmodels.api as sm

print("pandas", pd.__version__)
print("setup ok")

pandas 2.2.2
setup ok


## Load the data

One row per Summit Gear B2B customer, 2,200 rows, read straight from the course data
repo so it runs on Colab with no upload.

In [7]:
DATA_URL = ("https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-summer/week-8/summit_gear_customers.csv")

df = pd.read_csv(DATA_URL)
print("customers:", df.shape)
df.head()

customers: (2200, 17)


,customer_id,region,segment,industry,tenure_months,credit_limit,avg_balance,annual_spend,prior_late_count,days_since_last_order,orders_per_year,discount_rate,avg_order_value,noise_a,noise_b,days_to_pay,defaulted
0,C-01000,Mountain Central,Outdoor Retail,Retail,23,13914.12,4841.35,25992.60,3,41,13,6.44,1999.43,37.677,99.031,25.7,1
1,C-01001,Desert Southwest,Resort,Hospitality,51,65218.40,36656.53,393988.72,6,6,12,6.14,32832.39,47.640,0.247,71.2,0
2,C-01002,Desert Southwest,School District,Public Sector,3,19248.51,10015.56,69501.87,2,5,9,0.69,7722.43,49.448,64.429,46.5,0
3,C-01003,Desert Southwest,Resort,Hospitality,17,20349.31,8584.06,59762.60,2,108,15,0.00,3984.17,41.005,31.907,35.2,0
4,C-01004,Mountain Central,Outdoor Retail,Retail,2,33551.54,13782.11,96536.26,2,17,9,4.48,10726.25,21.311,70.355,37.5,0


## Look before you leap

Before you trust a model, look at the data: the column types, the default base rate (the
share of accounts that defaulted), and a quick summary of the payment-timing target.

In [8]:
print(df.dtypes)
print()
print("defaulted base rate:", round(df["defaulted"].mean(), 4))
print()
print(df["days_to_pay"].describe())

customer_id               object
region                    object
segment                   object
industry                  object
tenure_months              int64
credit_limit             float64
avg_balance              float64
annual_spend             float64
prior_late_count           int64
days_since_last_order      int64
orders_per_year            int64
discount_rate            float64
avg_order_value          float64
noise_a                  float64
noise_b                  float64
days_to_pay              float64
defaulted                  int64
dtype: object

defaulted base rate: 0.1991

count    2200.000000
mean       38.884273
std        17.978846
min         3.900000
25%        27.300000
50%        36.150000
75%        46.700000
max       200.100000
Name: days_to_pay, dtype: float64


## The predictor set

These are the numeric candidate predictors. Every one is a customer attribute known at
decision time, so none of them leaks the outcome. You use all of them first (the "kitchen
sink"), then prune in Exercise 2.

In [4]:
NUMERIC_PREDICTORS = [
    "credit_limit", "avg_balance", "annual_spend", "prior_late_count",
    "days_since_last_order", "orders_per_year", "discount_rate",
    "avg_order_value", "noise_a", "noise_b",
]
print(len(NUMERIC_PREDICTORS), "candidate predictors")

10 candidate predictors


# Exercise 1 — OLS for `days_to_pay`

**The question:** how long does a customer take to pay, and what drives it?

**What to do in this section:** fit one OLS regression of `days_to_pay` on the full
predictor set, confirm it ran, then read two coefficients and R-squared in the answer cell.
Below, do it in three moves:

**Specify.** The target is `days_to_pay`. For this first fit use the full predictor set
above — the "kitchen sink," every numeric predictor. Fitting it first, on purpose, sets up
the collinearity problem you solve in Exercise 2.

**Let the AI write the fit.** In the cell below, paste the `statsmodels` OLS the tutor
writes for you. Name the fitted results object exactly `ols_result`.

> **Package hint:** use `statsmodels` for OLS. The shape is
> `import statsmodels.api as sm; sm.OLS(y, sm.add_constant(X)).fit()` — here `y` is the
> `days_to_pay` column and `X` is the DataFrame of predictors. `add_constant` adds the
> intercept term.

In [5]:
# TODO (Exercise 1) — have the AI write this fit, then paste it here.
# Ask the AI tutor on the Week 8 lab page:
#   "Write the statsmodels OLS that regresses days_to_pay on credit_limit,
#    avg_balance, annual_spend, prior_late_count, days_since_last_order,
#    orders_per_year, discount_rate, avg_order_value, noise_a, noise_b, and
#    print the coefficient table with R-squared and adjusted R-squared."
# Name the fitted results object exactly `ols_result` (the Final check reads
# `ols_result.rsquared`). Then read off R-squared and adjusted R-squared.
ols_result = None   # replace None with the fitted model

                            OLS Regression Results                            
Dep. Variable:            days_to_pay   R-squared:                       0.844
Model:                            OLS   Adj. R-squared:                  0.843
Method:                 Least Squares   F-statistic:                     1185.
Date:                Fri, 10 Jul 2026   Prob (F-statistic):               0.00
Time:                        12:59:13   Log-Likelihood:                -7433.0
No. Observations:                2200   AIC:                         1.489e+04
Df Residuals:                    2189   BIC:                         1.495e+04
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    23.43

**Validate.** Confirm the model fit on all 2,200 rows and used the predictors you named.
Read off R-squared and adjusted R-squared.

**Interpret (this is the graded part).** In the answer cell below, write a short paragraph:

- Read **two** coefficients in collections terms, in the form "holding the other
  predictors fixed, a one-unit increase in X changes predicted `days_to_pay` by b days."
  Use `prior_late_count` and one other.
- State what R-squared says about how much of the in-sample variation in payment timing
  the model explains, in plain words. (These are in-sample numbers, not a claim about new
  customers.)
- Name one coefficient whose sign or size looks **wrong or unstable**, and say you will
  return to it in Exercise 2.

**Your interpretation (Exercise 1) — write here:**

> _Replace this line with your paragraph. Read two coefficients (one is `prior_late_count`)
> in collections terms, state what R-squared means in-sample, and name the coefficient that
> looks unstable._

## Helper — `predicted_days`

Score one customer from a fitted model by hand: the intercept plus each coefficient times
its predictor value — the same arithmetic the AI's `.predict()` does, written once so you
know what it is doing.

The docstring below is the **contract**: a hidden testing function calls `predicted_days`
by name with held-out inputs, so the parameter names and the return shape have to match
exactly. Keep the `def` line as given, write the body, and keep the docstring.

- **Takes:** `coefs` — a dict `{"const": intercept, "<predictor>": coefficient, ...}`; and
  `row` — a dict or pandas Series mapping predictor name to value.
- **Returns:** a **float**, the predicted `days_to_pay`: `coefs["const"]` plus, for each
  predictor in `row`, `coefs[name] * row[name]`.
- **Example:** `predicted_days({"const": 20.0, "prior_late_count": 3.0}, {"prior_late_count": 4})`
  is `32.0`.

> **Prompt card:** *"Explain how a fitted OLS turns a coefficient dict and one customer's
> values into a single predicted number."* Then write the body yourself.

In [ ]:
def predicted_days(coefs, row):
    """Predict days_to_pay for one customer from fitted OLS coefficients.

    Takes:
      coefs -- dict {"const": intercept, "<predictor>": coefficient, ...}
               (the "const" key holds the intercept).
      row   -- a dict or pandas Series mapping each predictor name to its value.
    Returns:
      float -- the predicted days_to_pay: coefs["const"] plus, for every predictor
               name in row, coefs[name] * row[name].
    """
    return None

In [ ]:
# See predicted_days work on a small example (independent of the fit above):
demo_coefs = {"const": 20.0, "prior_late_count": 3.0, "discount_rate": -1.5}
demo_row   = {"prior_late_count": 4, "discount_rate": 6}
print("predicted days:", predicted_days(demo_coefs, demo_row))   # 20 + 3*4 - 1.5*6 = 23.0

# Exercise 2 — feature selection: spot and drop a collinear predictor

**The problem you set up in Exercise 1.** Three predictors — `credit_limit`,
`avg_balance`, `annual_spend` — describe the same thing (account size) in three units.
When predictors move together, the model cannot tell their effects apart, so their
coefficients become **unstable**: inflated standard errors, sometimes a sign that makes no
business sense. This is **multicollinearity**, and diagnosing it is the key judgment in
this lab.

**Diagnose with a correlation matrix.** Ask the AI for the correlation matrix of the
numeric predictors and read the off-diagonal cells: a cell close to **1.0** (or **-1.0**)
between two different predictors is the warning sign. You will see `credit_limit`,
`avg_balance`, and `annual_spend` all correlate above 0.95 with each other. The `noise_a`
and `noise_b` columns correlate with nothing — not collinear, just useless, a different
problem you also fix. Name the fitted matrix exactly `corr`.

In [ ]:
# TODO (Exercise 2) — have the AI write the correlation matrix, then paste it here.
# Ask the AI tutor:
#   "Give me the correlation matrix of the numeric predictors in NUMERIC_PREDICTORS
#    as a DataFrame, and print it rounded to two decimals."
# Name the correlation DataFrame exactly `corr`. Read the off-diagonal cells: a value
# near 1.0 (or -1.0) between two DIFFERENT predictors is the warning sign.
corr = None   # replace None with the correlation matrix

## Helper — `flag_collinear_pairs`

The diagnosis as a function: read the correlation matrix and name the predictor pairs that
move together.

The docstring below is the **contract**: a hidden testing function calls
`flag_collinear_pairs` by name with a held-out correlation DataFrame, so the parameter and
the return shape have to match exactly. This helper takes the correlation **DataFrame**
directly — no reshaping step.

- **Takes:** `corr` — a pandas correlation DataFrame, exactly what `df[predictors].corr()`
  returns (square, with matching row and column labels); and `threshold`, default `0.9`.
- **Returns:** a **list of `(col_a, col_b, r)` tuples**, one per unique **upper-triangle**
  pair whose **absolute** correlation `abs(r)` is **at least** the threshold. `col_a` and
  `col_b` are predictor names (strings), `r` is their correlation (float). Use `abs(r)` so a
  strong negative correlation is caught too.
- **Example:** with `threshold=0.9`, a pair at `0.98` is returned; a pair at `0.20` is not;
  a pair at `-0.95` **is** returned (its absolute value is 0.95).

> **Prompt card:** *"Why does a correlation near 1.0 between two predictors make their
> individual regression coefficients unreliable?"* Then write the body yourself. Read only
> the upper triangle (`i, j` with `j > i`) so each pair is counted once and the diagonal
> `1.0` self-correlations are skipped.

In [ ]:
def flag_collinear_pairs(corr, threshold=0.9):
    """Flag predictor pairs that move together, from a correlation matrix.

    Takes:
      corr      -- a pandas correlation DataFrame (e.g. df[predictors].corr()),
                   square, with matching row and column labels.
      threshold -- float, default 0.9.
    Returns:
      list of (col_a, col_b, r) tuples -- one per UNIQUE upper-triangle pair whose
      absolute correlation abs(r) is at least threshold. col_a and col_b are the two
      predictor names (strings); r is their correlation (float).
    """
    return None

In [ ]:
# See flag_collinear_pairs run on the REAL correlation matrix.
if corr is not None:
    flagged = flag_collinear_pairs(corr, threshold=0.9)
    if flagged:
        print("pairs at or above 0.9:")
        for a, b, r in flagged:
            print("  {} & {}: {}".format(a, b, round(r, 3)))
    else:
        print("flag_collinear_pairs returned nothing yet — fill it in.")
else:
    print("Build the correlation matrix above (name it `corr`) first.")

## Decide and re-fit

Reduce the account-size cluster to **one** representative predictor (`avg_balance` is the
natural keep — it is the receivable actually outstanding) and drop the noise columns. Ask
the AI to re-fit the OLS on your reduced set. Name the fitted results object exactly
`ols_pruned`.

In [ ]:
# TODO (Exercise 2) — reduce the account-size cluster to ONE predictor, drop the
# noise columns, and have the AI re-fit the OLS on your reduced set. Paste it here.
# Ask the AI tutor:
#   "Re-fit the statsmodels OLS of days_to_pay on avg_balance, prior_late_count,
#    days_since_last_order, orders_per_year, discount_rate, and print the coefficient
#    table with R-squared and adjusted R-squared."
# Name the fitted results object exactly `ols_pruned` (the Final check reads
# `ols_pruned.rsquared`).
ols_pruned = None   # replace None with the pruned model

**Justify (this is the graded part).** In the answer cell below, write a short paragraph:

- Which predictors were collinear, and what in the correlation matrix told you (name the
  hot cells)?
- Which one did you **keep**, and why one is enough?
- What happened to **adjusted R-squared** when you dropped the redundant and noise columns,
  and why dropping them was right even though plain R-squared barely moved. (More
  predictors is not a better model; adjusted R-squared is the number that stops rewarding
  clutter.)

**Your justification (Exercise 2) — write here:**

> _Replace this line with your paragraph. Name the collinear cluster and the hot cells,
> say which one you kept and why one is enough, and explain what adjusted R-squared did
> when you dropped the redundant and noise columns._

# Exercise 3 — logistic for `defaulted`: odds, probabilities, one cutoff

**The question changes shape.** "Will this receivable go bad" is a yes / no, not a dollar
amount, so OLS is the wrong tool — a linear model would predict probabilities below 0 and
above 1. **Logistic regression** is built for a 0/1 outcome: it runs a linear score
through a **sigmoid** that squashes any number into a probability between 0 and 1.

**A short on-ramp.**
- A **probability** is a number from 0 to 1: the chance the outcome is 1.
- **Odds** are `p / (1 - p)`. A probability of 0.2 is odds of 0.25, or "1 to 4".
- A logistic coefficient is read on the log-odds scale, which is awkward raw, so we report
  `exp(coef)`, the **odds ratio**: the multiplier on the odds of default per one-unit rise
  in that predictor, holding the others fixed. Above 1 raises the odds of default; below 1
  lowers them.

**Specify.** The target is `defaulted`. Use the clean payment-behavior predictors:
`prior_late_count`, `days_since_last_order`, `discount_rate`, `orders_per_year` — each
known at decision time.

**Let the AI write the fit.** Paste the `statsmodels` logistic (Logit) the tutor writes,
printing the coefficient table and the odds ratios `exp(coef)`. Name the result exactly
`logit_result`.

> **Package hint:** use `statsmodels` `Logit` — `sm.Logit(y, sm.add_constant(X)).fit()` —
> or, if you prefer, `scikit-learn`'s `LogisticRegression`
> (`from sklearn.linear_model import LogisticRegression`). Either fits the same model.
> Name the fitted object exactly `logit_result`. Note the Final check reads
> `logit_result.params["prior_late_count"]`, which is the `statsmodels` shape; if you go the
> `scikit-learn` route that one scoreboard row may stay red even when your model is right,
> which is fine — the graded work is your interpretation and the three helper functions.

In [ ]:
# TODO (Exercise 3) — have the AI write the logistic fit, then paste it here.
# Ask the AI tutor:
#   "Write the statsmodels logistic regression (Logit) of defaulted on
#    prior_late_count, days_since_last_order, discount_rate, orders_per_year, and
#    print the coefficient table plus the odds ratios exp(coef)."
# Name the fitted results object exactly `logit_result` (the Final check reads its
# prior_late_count odds ratio).
logit_result = None   # replace None with the fitted logistic model

## Helper — `sigmoid`

The function that turns a linear score into a probability — the engine of logistic
regression, written once by hand.

The docstring below is the **contract**: a hidden testing function calls `sigmoid` by name
with held-out numbers, so the parameter and the return shape have to match exactly.

- **Takes:** `z` — a number (the linear score / log-odds).
- **Returns:** a **float**, `1 / (1 + exp(-z))`, a probability strictly between 0 and 1. Use
  `math.exp`.
- **Example:** `sigmoid(0)` is `0.5`; `sigmoid(2)` is about `0.881`.

> **Prompt card:** *"Why does the sigmoid guarantee an output between 0 and 1, and what is
> `sigmoid(0)`?"* Then write the body yourself.

In [ ]:
def sigmoid(z):
    """Map a linear score to a probability with the logistic function.

    Takes:
      z -- a number (the linear score / log-odds).
    Returns:
      float -- 1 / (1 + exp(-z)), a probability strictly between 0 and 1.
               sigmoid(0) is 0.5. Use math.exp.
    """
    return None

In [ ]:
# See sigmoid work:
print("sigmoid(0)  =", sigmoid(0))    # 0.5
print("sigmoid(2)  =", sigmoid(2))    # about 0.881
print("sigmoid(-2) =", sigmoid(-2))   # about 0.119

## Predicted probabilities and one cutoff

Read the odds ratios, then look at what the model predicts for a low-risk and a high-risk
example customer, and at **one** confusion matrix at a 0.5 cutoff.

**Pick a cutoff (and stop there).** A probability is not yet a decision. To act on it you
choose a **cutoff**: call everyone above it "predict default." Use **0.5** to start, then
read the one confusion matrix. Write **one sentence**: at a 0.5 cutoff, does the model
catch most defaults or miss most of them? (You will find it misses most — defaults are
only about 20 percent of the data, so 0.5 is a high bar. Choosing a cost-aware cutoff is
Week 9's job. Do not go past this one matrix.)

In [ ]:
# TODO (Exercise 3) — have the AI write the predicted probabilities for one low-risk
# and one high-risk example customer, and ONE confusion matrix at a 0.5 cutoff.
# Ask the AI tutor:
#   "Using logit_result, print the predicted default probability for a low-risk and a
#    high-risk example customer, then print one confusion matrix of actual vs predicted
#    defaulted at a 0.5 probability cutoff."
# Paste the code here, read the matrix once, then write your one-sentence cutoff read in
# the answer cell below.

**Interpret (this is the graded part).** In the answer cell below:

- Read the `prior_late_count` **odds ratio** in plain words: each additional prior late
  payment multiplies the odds of default by about how much?
- For your low-risk and high-risk example customers, state the **predicted probability**
  and say whether the ordering matches business intuition.
- Note the base rate of default (about 20 percent) and say why a probability is more useful
  to a collections manager than a flat yes / no.
- End with your **one-sentence 0.5-cutoff read**.

**Your interpretation (Exercise 3) — write here:**

> _Replace this line with your paragraph. Read the `prior_late_count` odds ratio, give the
> predicted probabilities for your two example customers and whether the ordering makes
> sense, note the base rate, and end with your one-sentence 0.5-cutoff read._

# Final check

This is your single check. **Run the cell below.** It runs a held-out test for each of the
six auto-checked pieces — the three helper functions and the three model fits — and tells
you, per piece, whether it is **exactly right** or a **little off**. There is a small numeric
tolerance, so an answer that is very close still counts. Each piece shows a green check for
done or a red x plus **the exercise to fix**. Use the **Table of Contents** (the list icon at
the top-left in Colab) to jump to a section. Re-run it any time. (Run *all* the cells above
first so your functions, models, and data are loaded.)

In [ ]:
# ===== RUN THIS to check your work. (You do not edit this cell.) =============
from IPython.display import display, HTML
import numpy as np
import pandas as pd


def final_check():
    """Check every auto-graded piece; show a green/red scoreboard and name the
    exercise to fix for anything not done yet."""

    def _close(got, want, tol):
        return isinstance(got, (int, float)) and abs(float(got) - want) <= tol

    def _check_predicted_days():
        got = predicted_days({"const": 5.0, "x": 2.0, "y": -3.0}, {"x": 10, "y": 2})
        return _close(got, 19.0, 1e-6)   # 5 + 2*10 + (-3)*2 = 19

    def _check_flag():
        # held-out correlation matrix: (a,b)=0.95 and (b,d)=-0.93 clear 0.9, nothing else does
        labels = ["a", "b", "c", "d"]
        m = pd.DataFrame(
            [[1.00, 0.95, 0.10, 0.20],
             [0.95, 1.00, 0.30, -0.93],
             [0.10, 0.30, 1.00, 0.40],
             [0.20, -0.93, 0.40, 1.00]],
            index=labels, columns=labels)
        got = flag_collinear_pairs(m, 0.9)
        names = {(t[0], t[1]) for t in got}
        return (names == {("a", "b"), ("b", "d")}
                and len(got) == 2
                and all(len(t) == 3 for t in got))

    def _check_sigmoid():
        return _close(sigmoid(0), 0.5, 1e-9) and _close(sigmoid(-2), 0.11920292, 1e-4)

    def _check_ols_full():
        return _close(ols_result.rsquared, 0.844, 0.01)

    def _check_ols_pruned():
        return _close(ols_pruned.rsquared, 0.844, 0.01)

    def _check_logit():
        return _close(float(np.exp(logit_result.params["prior_late_count"])), 1.87, 0.05)

    # (label, where-to-fix, test) -- each test returns True when that piece is done
    checks = [
        ("predicted_days",              "Exercise 1", _check_predicted_days),
        ("OLS fit (ols_result)",        "Exercise 1", _check_ols_full),
        ("flag_collinear_pairs",        "Exercise 2", _check_flag),
        ("pruned OLS fit (ols_pruned)", "Exercise 2", _check_ols_pruned),
        ("sigmoid",                     "Exercise 3", _check_sigmoid),
        ("logistic fit (odds ratio)",   "Exercise 3", _check_logit),
    ]

    rows, passed, todo = [], 0, []
    for label, where, test in checks:
        try:
            ok = bool(test())
        except Exception:
            ok = False
        if ok:
            passed += 1
            rows.append(
                "<tr><td style='font-size:18px'>&#9989;</td>"
                "<td style='padding:2px 14px'><code>{}</code></td>"
                "<td style='color:#176b3a'>complete</td></tr>".format(label))
        else:
            todo.append("{} ({})".format(label, where))
            rows.append(
                "<tr><td style='font-size:18px'>&#10060;</td>"
                "<td style='padding:2px 14px'><code>{}</code></td>"
                "<td style='color:#CC0000'>not done &#8212; fix in <b>{}</b></td></tr>".format(label, where))

    head = ("#176b3a" if passed == len(checks) else "#CC0000")
    nav = ("" if passed == len(checks) else
           "<p style='font-size:13px;color:#333;margin:8px 0 2px'>To fix these, scroll up to each "
           "exercise, or use the <b>Table of Contents</b> (the list icon at the top-left in Colab) to "
           "jump straight there. Then re-run this cell.</p>")
    html = (
        "<div style=\"font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif\">"
        "<h3 style='margin:4px 0;color:{}'>Lab 8 completeness check: {} / {} complete</h3>"
        "<table style='border-collapse:collapse;font-size:14px'>{}</table>{}"
        "<p style='font-size:12.5px;color:#666;margin-top:8px'>This checks one example each. Your grade also "
        "runs held-out inputs you do not see, so all-green here is necessary, not sufficient. Your written "
        "interpretations are read by the instructor, not auto-checked here.</p></div>"
    ).format(head, passed, len(checks), "".join(rows), nav)
    display(HTML(html))

    print("Lab 8 completeness: {} / {} complete.".format(passed, len(checks)))
    if todo:
        print("Still to do:", "; ".join(todo))
    else:
        print("All auto-checked pieces complete. Make sure your written interpretations are filled in.")


final_check()
# ============================================================================

## Before you submit

- **Run all cells from the top** so the notebook carries its fitted tables and the Final
  check reflects your latest work, then `File -> Download -> Download .ipynb`.
- **The single thing you upload from this notebook is the `.ipynb`** — with every cell run
  top to bottom so the outputs are saved. That is all this notebook produces. The AI-built
  HTML write-up and your written assessment of it are done in the **Lab 8 quiz in Canvas**,
  not here.
- **Aim for an all-green Final check** on the six auto-checked pieces. Your written
  interpretations for the three exercises are read by the instructor, not auto-checked here.
- **Keep every `def` line and model-variable name exactly as given.** The grader and the
  Final check call `predicted_days`, `flag_collinear_pairs`, `sigmoid`, `ols_result`,
  `ols_pruned`, and `logit_result` by name; renaming one scores that slice zero.
- **AI writes the fits; you own the reading.** Let the tutor write every `statsmodels` line.
  The graded work is the interpretation, the feature-selection decision, and the cutoff read
  — and the tutor will not write those before you have taken a pass.